# Dark Matter Quantum Sim — Toy Models Overview

**Remnant Fieldworks Inc.** · series `dark-matter-quantum-sim`

> **Honest scope.** These simulations model the *mathematical structure* of dark matter
> candidate interactions using small (2–4 qubit) toy Hamiltonians. They do **not** detect
> dark matter, prove any candidate exists, or replace experimental searches. The value is
> methodological: preregistered, ProofRecord-governed quantum simulation.

This notebook explores the three preregistered toy experiments (DM-001 axion, DM-002 sterile
neutrino, DM-003 WIMP) using the exact classical simulations in `dm_sim`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np
import matplotlib.pyplot as plt
from dm_sim.classical_sim import simulate_axion, simulate_neutrino, simulate_wimp
from dm_sim.metrics import match_threshold, verdict
print('dm_sim loaded')

## DM-001 — Axion (Peccei–Quinn double-well)

`H = ω₀(Z⊗I + I⊗Z) + λ(X⊗X)`, observable `⟨σ_z⟩` on qubit 0.
Threshold: dominant period `T = 2π/ω₀ ± 5%`.

In [ ]:
ax = simulate_axion(omega_0=1.0, lam=0.3)
print('target period 2pi/w :', ax['target_period'])
print('measured period     :', ax['measured_period'])
rel = abs(ax['measured_period']-ax['target_period'])/ax['target_period']
print('relative error      : %.3f%%' % (rel*100))
print('VERDICT             :', verdict(match_threshold(ax['measured_period'], ax['target_period'], 0.05)))

plt.figure(figsize=(8,3))
plt.plot(ax['times'], ax['sigma_z'])
plt.axhline(0, color='k', lw=0.5)
plt.xlabel('time'); plt.ylabel(r'$\langle\sigma_z\rangle$'); plt.title('DM-001 axion oscillation signature')
plt.tight_layout(); plt.show()

## DM-002 — Sterile neutrino (2-flavor oscillation)

`H = (Δm²/4E)(−cos2θ·Z + sin2θ·X)`, maximal mixing θ = π/4, Δm² = 1 eV².
Analytic survival probability cross-checked against matrix exponentiation
(kill condition: agree within 1%).

In [ ]:
nu = simulate_neutrino(delta_m2=1.0, energy=1.0, theta=np.pi/4)
print('max |analytic - matrix| :', nu['max_abs_diff'])
print('kill condition (>1%)    :', 'TRIGGERED' if nu['max_abs_diff']>0.01 else 'clear')

plt.figure(figsize=(8,3))
plt.plot(nu['L'], nu['analytic'], label='analytic')
plt.plot(nu['L'], nu['matrix'], '--', label='matrix exp')
plt.xlabel('L/E (arb.)'); plt.ylabel(r'$P(\nu_\mu\to\nu_\mu)$'); plt.legend()
plt.title('DM-002 sterile neutrino survival probability')
plt.tight_layout(); plt.show()

## DM-003 — WIMP–nucleon (exchange coupling)

`H = g(X⊗X + Y⊗Y)`, initial `|01⟩`, observable spin-flip `P(|01⟩→|10⟩)`.
Closed form `P = sin²(2gt)`; quarter period `t = π/(4g)` should give P = 1 (±5%).

In [ ]:
w = simulate_wimp(g=1.0)
print('max |sim - closed form| :', w['max_abs_diff'])
print('t_quarter = pi/(4g)     :', w['t_quarter'])
print('flip prob at quarter    :', w['flip_at_quarter'], '(target 1.0)')
print('VERDICT                 :', verdict(match_threshold(w['flip_at_quarter'], 1.0, 0.05), w['max_abs_diff']>0.01))

plt.figure(figsize=(8,3))
plt.plot(w['times'], w['flip'], label='simulated')
plt.plot(w['times'], w['analytic'], '--', label=r'$\sin^2(2gt)$')
plt.axvline(w['t_quarter'], color='r', ls=':', label='quarter period')
plt.xlabel('time'); plt.ylabel('flip probability'); plt.legend()
plt.title('DM-003 WIMP spin-flip probability')
plt.tight_layout(); plt.show()

## Takeaway

All three toy models are exactly classically simulable and pass their preregistered
kill conditions. The next step — for a physics collaborator — is to replace these toy
Hamiltonians with more realistic ones while keeping the same preregistration + ProofRecord
discipline.

*Proof before power.*